# Neural machine translation (Aplications of Natural Language Processing)

In this colab notebook you will learn how to work with a pre-trained model from [Huggingface](https://huggingface.co/) and how to fine tune such a model to improve its performace on your texts.

----
The code in this colab notebook is insprired in the content of the following websites:
* https://medium.com/@tskumar1320/how-to-fine-tune-pre-trained-language-translation-model-3e8a6aace9f
* https://huggingface.co/docs/transformers/training
* https://huggingface.co/docs/transformers/main_classes/tokenizer
* https://huggingface.co/docs/evaluate/


## Install the required libraries

In [ ]:
! pip install transformers[torch,sentencepiece]
! pip install datasets
! pip install evaluate
! pip install sacremoses
! pip install sacrebleu

## Give access to your Google Drive and set some variables

List of variables to set:
* `mydrive`: Full path to the folder in Google Drive where the corpora to be used for fine tuning is located
* `source`: Source language (see [ISO-639-1 two-letter codes](https://en.wikipedia.org/wiki/List_of_ISO_639-1_codes))
* `target`: Target language (see [ISO-639-1 two-letter codes](https://en.wikipedia.org/wiki/List_of_ISO_639-1_codes))
* `corpus`: Prefix of the files with the parallel corpus in moses format (two documents with the same number of lines; no blank line in either document, no duplicated parallel entries)
* `model_name`: Name of the pre-trained MT model to be used. You can use [any of the models](https://huggingface.co/Helsinki-NLP) made available by the NLP Language Technology Research Group at the University of Helsinki. Other models could also be used.
* `output_model_name`: Folder where the model after fine tuning will be saved.
* `train_size`: Amount of parallel sentences to be used for training (fine tuning).
* `test_size`: Amount of parallel sentences to be used for testing.
* `dev_size`: Amount of parallel sentences to be used for development.
* `patience`: Patience to be used for early stoppping.
* `batch_size`: Number of training samples to be used in each training step.

Note that the parallel corpus to be used for fine tuning must consist of at least `train_size`+`test_size`+`dev_size` parallel sentences.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

mydrive="/content/drive/MyDrive/anlp"
source = "en"
target = "es"
corpus = "News-Commentary.en-es.dedup"
model_name = "Helsinki-NLP/opus-mt-en-es"
output_model_name = "fine-tuned-model-en-es"

train_size = 1000
test_size = 200
dev_size = 200
patience = 3
batch_size = 64 ## Using fp16; otherwise use 32

source_path = mydrive + "/" + corpus + "."+source
target_path = mydrive + "/" + corpus + "."+target
#save_path = "/content/corpus-for-finetuning"

## Translate a sentence with the selected model

* The text to be translated must be assigned to the variable `text`

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

text = "This is a very simple example of translation from English into Spanish."

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda:0") # Use GPU 0

input_ids = tokenizer.encode(text, return_tensors="pt").to("cuda:0") # Use GPU 0
outputs = model.generate(input_ids)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(decoded)

del tokenizer
del model

## Generate a dataset from a parallel corpus in moses format
The moses format consists of two separate files with the same amount of lines so that the segments in the *n*-th line in both documents are mutual translation. The parallel corpus must not contain blank or duplicated entries.

Once created, the dataset is split into training, development and testing.

In [ ]:
from datasets import Dataset

# Read data from the files
with open(source_path, 'r', encoding='utf-8') as source_file, open(target_path, 'r', encoding='utf-8') as target_file:
    source_data = source_file.readlines()
    target_data = target_file.readlines()

# Create the dataset
dataset = Dataset.from_dict({
    'translation': [{source: src.strip(), target: tgt.strip()} for src, tgt in zip(source_data, target_data)]
})

print(dataset[0])  # Print the first pair of sentences
#dataset.save_to_disk(save_path)

# Split the dataset into training, development and testing
remaining_size = len(dataset) - test_size - dev_size
if (train_size > remaining_size):
    train_size = remaining_size

dataset_split = dataset.train_test_split(test_size=(test_size+dev_size), train_size=train_size)
train_dataset = dataset_split['train']
test_dev_dataset = dataset_split['test']

dataset_split = test_dev_dataset.train_test_split(test_size=dev_size, train_size=test_size)
dev_dataset = dataset_split['test']
test_dataset = dataset_split['train']

print("Size of the training set: "+str(len(train_dataset)))
print("Size of the development set: "+str(len(dev_dataset)))
print("Size of the test set: "+str(len(test_dataset)))

## Translate the test set using the pre-trained model



In [ ]:
from transformers import pipeline
import evaluate

# Get the sentences in the test set
inputs = [ex[source] for ex in test_dataset["translation"]]
references = [ex[target] for ex in test_dataset["translation"]]

# Translate using pipelines - Use GPU 0 (device="cuda:0")
translator = pipeline("translation", model=model_name, device="cuda:0", batch_size=64)
pre_outputs = translator(inputs)
outputs = [ex["translation_text"] for ex in pre_outputs]

metric = evaluate.load("sacrebleu") # BLEU
result = metric.compute(predictions=outputs, references=references)
print (result)

del translator

## Preprocess the datasets before their use for fine tuning
The proprocessing implies tokenizing the sentences in the datasets using the tokenizer included in the pre-trained model


In [ ]:
from transformers import AutoTokenizer

max_input_length = 128
max_target_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = [ex[source] for ex in examples["translation"]]
    targets = [ex[target] for ex in examples["translation"]]
    model_inputs = tokenizer(text=inputs, max_length=max_input_length, padding=True, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, padding=True, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_dev_dataset = dev_dataset.map(preprocess_function, batched=True)

## Fine tune the model on the dataset created above
Before fine tuning, we need to set the automatic evaluation metric to be used to evaluate on the development set, then we will run the training algorithm on the training dataset


### Define the metric to be used on the development set

In [ ]:
import evaluate

metric = evaluate.load("sacrebleu") # BLEU

import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

### Fine tune the model

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda:0") # Load in GPU 0

args = Seq2SeqTrainingArguments(
    output_dir="./"+output_model_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    #evaluation_strategy="steps",
    #save_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=30,
    predict_with_generate=True,
    fp16=True,
    metric_for_best_model="bleu",
    load_best_model_at_end=True, # It uses metric_for_best_model to compare models
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_dev_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)], # It uses metric_for_best_model
)

trainer.train()
trainer.save_model()

del trainer
del model
del data_collator

## Translate the test set using the fine-tuned model

In [ ]:
from transformers import pipeline
import evaluate

# Get the sentences in the test set
inputs = [ex[source] for ex in test_dataset["translation"]]
references = [ex[target] for ex in test_dataset["translation"]]

# Translate using pipelines - Use GPU 0 (device="cuda:0")
translator = pipeline("translation", model=output_model_name, device="cuda:0", batch_size=64)
pre_outputs = translator(inputs)
outputs = [ex["translation_text"] for ex in pre_outputs]

metric = evaluate.load("sacrebleu") # BLEU
result = metric.compute(predictions=outputs, references=references)
print (result)

del translator